# Transaction Anomaly Detection

Implements the first analytical feature of Finance Analytics: detecting
transactions whose amount is unusually different from the user's own
historical behaviour.

**"Anomaly" is deliberately narrow here.** It does **not** mean fraud,
incorrect spending, or financial harm — it means *statistically unusual
relative to this account's own past transactions*. A large transaction that
matches a category's established pattern (a Travel category with a history
of €500-650 purchases) is not an anomaly. A transaction that breaks an
otherwise stable pattern (a merchant that has charged the exact same
subscription amount every time) is — even if the new amount itself isn't
large in absolute terms.

**Why this is useful.** Understanding *which* transactions look unusual —
and *why*, in plain language — is a prerequisite for any later insight or
recommendation feature (PR-009's Product Boundary explicitly keeps this PR
to the analytical result only: no Android display, no LLM narrative, no
recommendations, no persistence yet).

Built directly on `notebooks/02_exploratory_data_analysis.ipynb` — this
notebook does not re-derive EDA findings, it applies them.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from finance_analytics.analysis.outliers import (
    flag_category_relative_outliers,
    flag_iqr_outliers,
    robust_zscores,
)
from finance_analytics.anomalies.detector import (
    ANOMALY_Z_THRESHOLD,
    MIN_CATEGORY_HISTORY,
    MIN_GLOBAL_HISTORY,
    MIN_MERCHANT_HISTORY,
    detect_anomalies,
)
from finance_analytics.anomalies.features import build_historical_features
from finance_analytics.io.csv import load_transactions_csv

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_PATH = Path("../data/raw/finance_analytics_test_transactions.csv")

## Dataset

Same fixture and same cleaning steps as notebook 02: drop rows with an
invalid date/amount, drop the duplicate transaction ID, and drop the QA
fixture row disguised as a transaction (`description` containing "test").
See notebook 02, section 3, for the full reasoning — repeated here only
because this notebook needs the same clean, deduplicated input.

In [2]:
transactions = load_transactions_csv(DATA_PATH)

structurally_valid = transactions.dropna(subset=["date", "amount"]).drop_duplicates(subset=["id"])
synthetic_qa_mask = structurally_valid["description"].str.contains("test", case=False, na=False)
clean = structurally_valid.loc[~synthetic_qa_mask].copy()

print(f"Clean rows: {len(clean)}")
clean.head()

Clean rows: 18


,id,date,amount,currency,description,merchant,category,account
0,1,2026-02-02,-12.50,EUR,Morning coffee,Coffee Corner,Food & Dining,Main Account
1,2,2026-02-03,-54.90,EUR,Weekly groceries,Continente,Groceries,Main Account
2,3,2026-02-05,-29.99,EUR,Monthly subscription,Spotify,Subscriptions,Main Account
3,4,2026-02-07,-82.40,EUR,Dinner with friends,O Pescador,Food & Dining,Main Account
4,5,2026-02-10,-45.00,EUR,Electricity bill,EDP,Utilities,Main Account


## 1. What the EDA Revealed

From `02_exploratory_data_analysis.ipynb`, section 9 (Outlier Investigation)
and its Key Findings:

- **A naive global check on signed amount misclassifies income.** Flagging
  the most extreme values of `amount` directly caught `Employer Payroll`'s
  €3,450 salary alongside genuine large purchases — income and expenses
  don't belong on the same scale. Any method used here must be scoped to
  expenses.
- **A global robust z-score on expense magnitude worked well.** It cleanly
  separated `MediaMarkt` (z ≈ 24) and `TAP Air` (z ≈ 17) from a next tier
  sitting at z ≈ 1-2 — a strong, well-separated signal on *this* dataset,
  using the median/MAD estimator that PR-008 chose specifically because it
  isn't itself distorted by the outliers it's measuring.
- **Category-relative IQR returned nothing.** Not because nothing was
  unusual, but because most categories have only 1-3 transactions here —
  too few to define IQR fences at all. EDA's own conclusion: "an empty
  result from this method, on this dataset, is a statement about sample
  size, not a clean bill of health."
- **Most merchants appear exactly once.** Only `Spotify`, `Continente` and
  `O Pescador` repeat (each twice) — the only merchants a merchant-relative
  baseline could say anything about in this window at all.
- **Expense amounts are strongly right-skewed** (mean €136.30 vs median
  €41.40), driven by two one-off purchases — any threshold naive to that
  skew (e.g. mean ± std) would be misleading.

The method chosen below is a direct response to all five points.

## 2. Methods Considered

Re-running the three PR-008 approaches on this notebook's `clean` data,
purely to keep the comparison concrete and quantified in this notebook
rather than only narrated:

In [3]:
expenses = clean.loc[clean["amount"] < 0].copy()
expenses["abs_amount"] = expenses["amount"].abs()

comparison = pd.DataFrame(
    {
        "method": [
            "Global IQR, signed amount",
            "Global robust z-score, expense magnitude",
            "Category-relative IQR",
        ],
        "flagged_count": [
            int(flag_iqr_outliers(clean["amount"]).sum()),
            int((robust_zscores(expenses["abs_amount"]) > 3).sum()),
            int(flag_category_relative_outliers(clean).sum()),
        ],
        "verdict": [
            "Rejected — flags income alongside genuine large purchases.",
            (
                "Works, but every transaction shares one nationwide-of-everything "
                "baseline regardless of category or merchant."
            ),
            (
                "Rejected as a sole method — degenerates to nothing when a "
                'category has too few points to define "usual".'
            ),
        ],
    }
)
comparison

,method,flagged_count,verdict
0,"Global IQR, signed amount",4,Rejected — flags income alongside genuine larg...
1,"Global robust z-score, expense magnitude",2,"Works, but every transaction shares one nation..."
2,Category-relative IQR,0,Rejected as a sole method — degenerates to not...


`sklearn.ensemble.IsolationForest` was not run at all: this fixture has 18
clean rows. There is no volume here to train, tune, or meaningfully
validate an ML model against, and PR-009 explicitly warns against reaching
for ML just because scikit-learn is part of the stack. It stays out of this
workspace's dependencies (see `analytics/README.md`) until a PR that
actually has the data to justify it.

**Comparing on the criteria PR-009 asks for:**

| Criterion | Global IQR (signed) | Global robust z (expense) | Category-relative IQR | IsolationForest |
|---|---|---|---|---|
| Interpretability | High | High | High (in principle) | Low — no per-feature reason |
| Stability (skewed spending) | Poor — assumes symmetry | Good — median/MAD is skew-robust | Good, when it has enough data | Unknown — unvalidatable at this size |
| Small sample sizes | N/A | Degrades gracefully (still a real number) | Degenerates to empty | Fails outright — nothing to fit |
| Category differences | Ignored | Ignored | Directly modelled | Could learn it, unverifiably |
| Explainability | Medium | Medium | High, when it works | Low |

Robust (median/MAD) z-scoring is the strongest primitive here on every axis
that matters for this dataset. The open problem is *what to score each
transaction against* — that's what section 4 resolves.

## 3. Selected Method and Rationale

**A historical, leakage-free robust z-score, scored against the most
specific baseline with enough history to trust:**

1. **Merchant-relative** — this merchant's own prior amounts.
2. **Category-relative** — this category's prior amounts.
3. **Global** — every prior expense, regardless of merchant/category.
4. **`insufficient_history`** — none of the above has enough data yet; the
   transaction is not scored at all, rather than scored against a baseline
   too thin to mean anything.

Each tier only ever looks at transactions *strictly before* the one being
scored (chronological order) — a transaction never contributes to its own
baseline. See `anomalies/features.py`.

In [4]:
print(f"Minimum merchant history to trust a merchant baseline: {MIN_MERCHANT_HISTORY}")
print(f"Minimum category history to trust a category baseline: {MIN_CATEGORY_HISTORY}")
print(f"Minimum global history to trust the global fallback:   {MIN_GLOBAL_HISTORY}")
print(f"Anomaly threshold (robust z-score):                    > {ANOMALY_Z_THRESHOLD}")

Minimum merchant history to trust a merchant baseline: 2
Minimum category history to trust a category baseline: 3
Minimum global history to trust the global fallback:   5
Anomaly threshold (robust z-score):                    > 3.0


**Why a hierarchy, and why these specific thresholds:**

- **Merchant first, with the lowest threshold (2).** EDA's recurring-transaction
  investigation found `Spotify` charging an *identical* amount both times it
  appeared — a tight, trustworthy baseline from just two observations,
  because what matters for a merchant baseline is *consistency*, not count.
  A category-tuned threshold would needlessly discard that signal.
- **Category second, with a higher threshold (3).** Category spend mixes
  several merchants and purchase types (a `Food & Dining` category here
  covers a €8 lunch and a €82 dinner), so a category median/MAD needs more
  points before it stops being dominated by one observation — matching
  EDA's own finding that 1-2-point category IQR is degenerate.
- **Global last, with the highest threshold (5).** It is the broadest,
  least specific baseline used only when nothing more targeted is
  available — it should require more history to be trusted than the finer
  tiers, precisely because it says the least about *this* transaction's own
  context.
- **Only the positive direction is flagged** (`score > 3.0`, not `|score| >
  3.0`). This product asks "is this transaction unusually *large* for its
  context" — PR-009's own explainability example ("3.8× higher than
  typical") is one-directional. An unusually *small* transaction isn't
  flagged; there's no evidence from the EDA or the product requirements
  that it should be.
- **3.0 as the threshold** mirrors the separation EDA observed on the
  global check: genuine standouts at z ≈ 17-24, everything else at z ≈ 1-2.
  It is a reasonable boundary given that evidence, not a value tuned or
  validated against labelled data — see Limitations.

## 4. Feature Engineering

Against the candidates EDA's "Implications for Feature Engineering" section
identified:

| Candidate | Used here? | Why |
|---|---|---|
| `amount` (magnitude) | Yes | The base signal every tier scores. |
| `log_amount` | No | Robust z-scoring already uses median/MAD, which is skew-resistant by construction (that's *why* PR-008 picked it over mean/std) — a log transform would add a step without adding robustness a scale-aware estimator doesn't already have, and would break the human-readable "×higher than typical" phrasing in section 5's explanations. |
| Category-relative amount | Yes | The category tier. |
| Merchant-relative amount | Yes | The merchant tier — highest priority in the hierarchy. |
| `merchant_frequency` | Yes, indirectly | Drives `merchant_history_count`, the gate for whether the merchant tier is trusted at all. |
| `category_frequency` | Yes, indirectly | Same role for the category tier. |
| `days_since_previous_transaction` | No | No EDA evidence connects transaction *timing* to whether an *amount* is unusual — that candidate is about cadence, which is what recurring-payment detection would use (explicitly out of scope for this PR). |

`build_historical_features` computes the raw ingredients (history counts,
medians, MADs) that `detect_anomalies` then picks between:

In [5]:
features = build_historical_features(clean)
features[
    [
        "id",
        "date",
        "merchant",
        "category",
        "abs_amount",
        "merchant_history_count",
        "merchant_median",
        "category_history_count",
        "category_median",
        "global_history_count",
        "global_median",
    ]
]

,id,date,merchant,category,abs_amount,merchant_history_count,merchant_median,category_history_count,category_median,global_history_count,global_median
0,1,2026-02-02,Coffee Corner,Food & Dining,12.50,0,NaN,0,NaN,0,NaN
1,2,2026-02-03,Continente,Groceries,54.90,0,NaN,0,NaN,1,12.500
2,3,2026-02-05,Spotify,Subscriptions,29.99,0,NaN,0,NaN,2,33.700
3,4,2026-02-07,O Pescador,Food & Dining,82.40,0,NaN,1,12.50,3,29.990
4,5,2026-02-10,EDP,Utilities,45.00,0,NaN,0,NaN,4,42.445
5,6,2026-02-12,Farmácia Central,Health,18.75,0,NaN,0,NaN,5,45.000
6,7,2026-02-15,CP,Transport,120.00,0,NaN,0,NaN,6,37.495
7,8,2026-02-18,TAP Air,Travel,642.50,0,NaN,0,NaN,7,45.000
8,9,2026-02-20,MediaMarkt,Shopping,899.00,0,NaN,0,NaN,8,49.950
9,10,2026-02-22,Café Central,Food & Dining,8.20,0,NaN,2,47.45,9,54.900


## 5. Detection Results

`detect_anomalies` runs the full hierarchy and returns one `AnomalyResult`
per expense transaction — `Income` rows aren't scored (see section 1: a
naive check that doesn't make this distinction misclassifies salary).

In [6]:
results = detect_anomalies(clean)

results_df = pd.DataFrame([r.__dict__ for r in results]).rename(columns={"transaction_id": "id"})
results_df = results_df.merge(
    clean[["id", "date", "merchant", "category", "amount"]], on="id", how="left"
)
results_df["abs_amount"] = results_df["amount"].abs()

results_df[
    [
        "id",
        "date",
        "merchant",
        "category",
        "abs_amount",
        "method",
        "anomaly_score",
        "is_anomaly",
        "reason",
    ]
]

,id,date,merchant,category,abs_amount,method,anomaly_score,is_anomaly,reason
0,1,2026-02-02,Coffee Corner,Food & Dining,12.50,insufficient_history,NaN,False,Not enough transaction history to evaluate thi...
1,2,2026-02-03,Continente,Groceries,54.90,insufficient_history,NaN,False,Not enough transaction history to evaluate thi...
2,3,2026-02-05,Spotify,Subscriptions,29.99,insufficient_history,NaN,False,Not enough transaction history to evaluate thi...
3,4,2026-02-07,O Pescador,Food & Dining,82.40,insufficient_history,NaN,False,Not enough transaction history to evaluate thi...
4,5,2026-02-10,EDP,Utilities,45.00,insufficient_history,NaN,False,Not enough transaction history to evaluate thi...
5,6,2026-02-12,Farmácia Central,Health,18.75,global_relative_robust_z,-1.18,False,Amount is close to your typical transaction ra...
6,7,2026-02-15,CP,Transport,120.00,global_relative_robust_z,3.08,True,Amount is 3.2× higher than your typical transa...
7,8,2026-02-18,TAP Air,Travel,642.50,global_relative_robust_z,15.35,True,Amount is 14.3× higher than your typical trans...
8,9,2026-02-20,MediaMarkt,Shopping,899.00,global_relative_robust_z,17.99,True,Amount is 18.0× higher than your typical trans...
9,10,2026-02-22,Café Central,Food & Dining,8.20,global_relative_robust_z,-0.87,False,Amount is close to your typical transaction ra...


In [7]:
fig = px.scatter(
    results_df.dropna(subset=["anomaly_score"]),
    x="abs_amount",
    y="anomaly_score",
    color="is_anomaly",
    hover_data=["merchant", "category", "method"],
    title="Transaction Amount vs Anomaly Score",
    labels={"abs_amount": "Transaction amount (EUR)", "anomaly_score": "Anomaly score (robust z)"},
)
fig.add_hline(y=ANOMALY_Z_THRESHOLD, line_dash="dot", annotation_text="anomaly threshold")
fig.show()

Rows dropped from this chart are the `insufficient_history` transactions —
they have no score to plot, which is itself the point (section 7).

In [8]:
fig = px.strip(
    results_df,
    x="category",
    y="abs_amount",
    color="is_anomaly",
    hover_data=["merchant", "date", "method"],
    title="Transaction Amounts by Category, Flagged Transactions Highlighted",
    labels={"abs_amount": "Transaction amount (EUR)", "category": ""},
)
fig.show()

A strip plot, not a box plot — most categories here have only 1-4
transactions, the same reason notebook 02 avoided box plots for category
distributions (a box plot on that few points looks precise while being
nearly meaningless).

In [9]:
fig = px.scatter(
    results_df.sort_values("date"),
    x="date",
    y="abs_amount",
    color="is_anomaly",
    size="abs_amount",
    hover_data=["merchant", "category", "method"],
    title="Transaction Timeline, Flagged Transactions Highlighted",
    labels={"abs_amount": "Transaction amount (EUR)", "date": ""},
)
fig.show()

## 6. Controlled Synthetic Validation

No ground-truth anomaly labels exist for this dataset (or for real user
data). Per PR-009: use controlled synthetic cases instead of claiming
accuracy. Two scenarios, matching the test suite in
`tests/test_anomalies_detector.py`:

**A — a large transaction that's normal for its category should not be
flagged.**

In [10]:
synthetic_travel = pd.DataFrame(
    [
        {
            "id": "s1",
            "date": pd.Timestamp("2026-01-01"),
            "amount": -500.0,
            "category": "Travel",
            "merchant": "TAP Air",
        },
        {
            "id": "s2",
            "date": pd.Timestamp("2026-01-08"),
            "amount": -650.0,
            "category": "Travel",
            "merchant": "Ryanair",
        },
        {
            "id": "s3",
            "date": pd.Timestamp("2026-01-15"),
            "amount": -600.0,
            "category": "Travel",
            "merchant": "TAP Air",
        },
        {
            "id": "s4",
            "date": pd.Timestamp("2026-01-22"),
            "amount": -620.0,
            "category": "Travel",
            "merchant": "Ryanair",
        },
    ]
)
scenario_a = next(r for r in detect_anomalies(synthetic_travel) if r.transaction_id == "s4")
scenario_a

AnomalyResult(transaction_id='s4', anomaly_score=0.27, is_anomaly=False, method='category_relative_robust_z', reason='Amount is close to your typical Travel transaction (usually around €600.00).', reference_context={'merchant_median': 650.0, 'merchant_transaction_count': 1, 'category_median': 600.0, 'category_iqr': 75.0, 'category_transaction_count': 3, 'global_transaction_count': 3})

A €620 flight, once Travel already has an established €500-650 history, is
unremarkable — exactly PR-009's requirement that "the detector should not
flag a transaction simply because it is large if that amount is normal for
its category."

**B — a small deviation from an otherwise perfectly consistent merchant
should be flagged, even with thin history.**

In [11]:
synthetic_subscription = pd.DataFrame(
    [
        {
            "id": "s1",
            "date": pd.Timestamp("2026-01-05"),
            "amount": -29.99,
            "category": "Subscriptions",
            "merchant": "Spotify",
        },
        {
            "id": "s2",
            "date": pd.Timestamp("2026-02-05"),
            "amount": -29.99,
            "category": "Subscriptions",
            "merchant": "Spotify",
        },
        {
            "id": "s3",
            "date": pd.Timestamp("2026-03-05"),
            "amount": -59.99,
            "category": "Subscriptions",
            "merchant": "Spotify",
        },
    ]
)
scenario_b = next(r for r in detect_anomalies(synthetic_subscription) if r.transaction_id == "s3")
scenario_b

AnomalyResult(transaction_id='s3', anomaly_score=inf, is_anomaly=True, method='merchant_relative_robust_z', reason='Amount is 2.0× higher than your typical Spotify transaction (usually around €29.99).', reference_context={'merchant_median': 29.99, 'merchant_transaction_count': 2, 'category_median': 29.99, 'category_iqr': 0.0, 'category_transaction_count': 2, 'global_transaction_count': 2})

Only two prior Spotify charges exist, both identical — nowhere near the
category threshold of 3, but the merchant tier's threshold of 2 is enough,
because a merchant that has never varied is a maximally tight baseline. The
€59.99 charge scores as an infinite deviation (`mad == 0`, see
`outliers.robust_zscore_of`'s docstring) — a doubled subscription price is
about as sharp a break from history as this method can express.

## 7. Known Examples

The two genuine large purchases PR-008 identified and manually inspected —
`TAP Air` (€642.50) and `MediaMarkt` (€899.00) — are used here as a
**validation case, not proof of general performance** (PR-009 §6).

In [12]:
known_examples = results_df.set_index("id").loc[["8", "9"]]
known_examples[["date", "merchant", "category", "abs_amount", "method", "anomaly_score", "reason"]]

,date,merchant,category,abs_amount,method,anomaly_score,reason
id,,,,,,,
8,2026-02-18,TAP Air,Travel,642.5,global_relative_robust_z,15.35,Amount is 14.3× higher than your typical trans...
9,2026-02-20,MediaMarkt,Shopping,899.0,global_relative_robust_z,17.99,Amount is 18.0× higher than your typical trans...


Both are flagged, both via `global_relative_robust_z` — at the time each
occurs, neither `Travel` nor `Shopping` nor their specific merchants have
any prior history, so the hierarchy correctly falls all the way back to the
global baseline. That the detector's definition of "anomaly" agrees with
these being flagged is expected, not surprising: PR-008 already established
these are genuine, well-formed one-off purchases (no duplicate row, no
mismatched currency, description matches merchant and category) — this
detector's job was never to say they're fraudulent, only that they're
unusual *relative to the account's history so far*, which they clearly are.

## 8. False-Positive Inspection

Two flagged transactions worth a closer, skeptical look — this is the part
of the spec that matters most in the absence of ground truth.

In [13]:
borderline = results_df.set_index("id").loc[["7", "15"]]
borderline[
    [
        "date",
        "merchant",
        "category",
        "abs_amount",
        "method",
        "anomaly_score",
        "reference_context",
        "reason",
    ]
]

,date,merchant,category,abs_amount,method,anomaly_score,reference_context,reason
id,,,,,,,,
7,2026-02-15,CP,Transport,120.0,global_relative_robust_z,3.08,"{'merchant_median': None, 'merchant_transactio...",Amount is 3.2× higher than your typical transa...
15,2026-03-08,O Pescador,Food & Dining,37.8,category_relative_robust_z,3.97,"{'merchant_median': 82.4, 'merchant_transactio...",Amount is 3.0× higher than your typical Food &...


**Transaction 15 — O Pescador, €37.80 dinner, flagged (`category_relative`,
z ≈ 3.97).** `Food & Dining`'s history at this point is `[€12.50 coffee,
€82.40 dinner (also O Pescador), €8.20 lunch]` — median €12.50, with the MAD
pulled up by the earlier €82.40 dinner. A modest €37.80 dinner reads as
"~3× typical" only because the category itself lumps three very different
purchase types (coffee, lunch, a proper dinner out) into one baseline. This
looks like a **false-positive risk from thin, heterogeneous category
history** rather than genuinely unusual behaviour — a second dinner isn't
surprising once the first one is already in the history that should have
softened the baseline, but three points isn't enough for the median/MAD to
reflect that yet.

**Transaction 7 — CP, €120 train tickets, flagged (`global_relative`, z ≈
3.08).** This sits right at the 3.0 threshold. With only 6 prior expense
transactions to compare against, the global median (€37.49) is itself a
rough estimate — a train ticket purchase being "anomalous" is genuinely
ambiguous at this sample size, not a confident call. This is the kind of
near-threshold result any downstream consumer of this detector should treat
with lower confidence than a z ≈ 15-18 case like section 7's examples.

Neither of these invalidates the method — both are exactly the kind of
honest limitation a small, non-ground-truthed dataset should surface, and
both are visible and explainable rather than silent.

## 9. Limitations

- **Dataset size.** 16 scored expense transactions (18 clean rows minus 2
  `Income` rows). Most transactions either fall to `insufficient_history` or
  the coarse global tier rather than the more specific, more explainable
  merchant/category tiers — the hierarchy is sound, but this fixture rarely
  has enough history to exercise its more specific branches.
- **No ground truth.** No labelled anomalies exist for this dataset or for
  real user data. Sections 6-8 substitute controlled synthetic cases and
  manual inspection for accuracy claims — no precision/recall figure is
  reported anywhere in this notebook, because none would be meaningful.
- **Cold start is unavoidable, by design.** The first several transactions
  of any account are always `insufficient_history`, regardless of how
  unusual they might be — the detector needs history before it can say
  anything, which is a defensible trade-off, not a bug, but worth stating
  plainly.
- **Thresholds are evidence-motivated, not statistically tuned.** `2` / `3`
  / `5` (minimum history) and `3.0` (anomaly z-score) were chosen by
  analogy to the separation EDA observed on one 18-row fixture. Whether
  they're well-calibrated for a different user's spending pattern is
  untested.
- **Category granularity affects the result**, as transaction 15 shows —
  `Food & Dining` mixing coffee/lunch/dinner into one baseline is a
  category-design question this PR cannot fix, only surface.
- **Only large deviations are flagged**, by deliberate scope decision (see
  section 3) — an unusually *small* transaction is never flagged here.
- **Synthetic/test data.** As in notebook 02: this fixture exists to
  exercise code paths, not to represent real spending behaviour.

## 10. Next Steps

- Re-evaluate the minimum-history thresholds and the `3.0` anomaly
  threshold once a full year of real (or more realistic synthetic) data
  exists, ideally against some form of manually-labelled review sample.
- Investigate whether splitting heterogeneous categories (e.g. `Food &
  Dining` by purchase type) reduces false positives like transaction 15.
- Revisit whether `sklearn.ensemble.IsolationForest` becomes justified once
  transaction volume grows enough to train and validate it properly.
- Consider, as a product question (not an engineering one for this PR),
  whether unusually *small* transactions deserve their own signal.
- The next analytical feature should build on this result set — but per
  PR-009's Product Boundary, this PR stops at the analytical result. No
  Android display, no LLM-generated narrative, no recommendations, and no
  persistence layer are implemented here.

## Out of Scope — Confirmation

Not implemented in this notebook or in `finance_analytics.anomalies`:
fraud detection, financial advice, LLM-generated explanations (every
`reason` string above is a fixed template — see `anomalies/explanations.py`),
recommendations, recurring-payment detection, forecasting, automated
categorisation, Android integration, and a production analytics API.